In [0]:
BASE_DIR = '/Volumes/sales/landing/sales/Products/'
CHECKPOINT_DIR = '/Volumes/sales/landing/sales/CheckPoint_Directory/'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/Products'
CATALOG = 'Sales'
SCHEMA = 'Bronze'
TARGET_TABLE = 'Products'
print(f'checkpoint_location : {CHECKPOINT_LOCATION}')
print(f'BASE_LOCATION = {BASE_DIR}')

In [0]:
Products_schema = ''' product_id STRING, 
                      product_category_name STRING, 
                      product_name_lenght DECIMAL, 
                      product_description_lenght DECIMAL, 
                      product_photos_qty DECIMAL, 
                      product_weight_g DECIMAL, 
                      product_length_cm DECIMAL, 
                      product_height_cm DECIMAL, 
                      product_width_cm DECIMAL'''

In [0]:
Products_df = spark.read.format('csv')\
                  .option('header', 'true')\
                  .schema(Products_schema)\
                  .load(f'{BASE_DIR}')
display(Products_df.limit(3))
Products_df.printSchema()

%md
#### LOAD THE `Products` DATA INTO BRONZE LAYER
> Order_Sales . Bronze . printSchema

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
Products_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(Products_schema)
                 .load(f'{BASE_DIR}')
        )
Products_trans_df = Products_df.select(
    "product_id", 
    "product_category_name", 
    "product_name_lenght", 
    "product_description_lenght", 
    "product_photos_qty", 
    "product_weight_g", 
    "product_length_cm", 
    "product_height_cm", 
    "product_width_cm",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

#Products_trans_df.printSchema()
#Products_trans_df.limit(3).display(checkpointLocation=f'{CHECKPOINT_DIR}/Products_display')

In [0]:
qry = (
    Products_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()